# Ngày 4 - Từ 17 cái chấm của bạn đến một model pose (Lane S)

Notebook này thực hiện luồng duy nhất:
1. Nhập **`REPO_URL`** (Link repository GitHub cá nhân của bạn, đã commit 20 file `.txt` nhãn train).
2. Upload file **`gold_labels.zip`** (Bộ nhãn chuẩn Golden Set do Instructor cấp).
3. Tự động kiểm tra nhãn & đánh giá OKS nhãn của bạn so với Gold Set (`outputs/eval_vs_gold.json`).
4. Fine-tune mô hình YOLO Pose (`yolo26n-pose.pt`) trên 20 ảnh nhãn của bạn.
5. Đánh giá mô hình trên tập test (`outputs/eval_model.json`).
6. Đóng gói tất cả file kết quả (`outputs.zip`) và tự động kích hoạt tải về máy local để làm báo cáo `reports/REPORT.md`.


In [ ]:
%pip -q install -U ultralytics

import json
import os
import sys
import subprocess
import zipfile
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

# =====================================================================
# 1. Nhập REPO_URL GitHub cá nhân của Học viên & Clone Repo
# =====================================================================
# @markdown ### 🔗 Link Repository GitHub cá nhân của Học viên (đã commit 20 file txt nhãn train)
REPO_URL = "https://github.com/VinUni-AI20k/Day4-TrackData-Keypoint-Pose.git" # @param {type:"string"}

CONTENT_DIR = Path('/content') if Path('/content').exists() else Path.cwd()

if not REPO_URL.strip():
    raise ValueError("⚠️ Vui lòng điền REPO_URL (Link repository GitHub cá nhân của bạn) vào ô trên trước khi chạy!")

repo_dir = CONTENT_DIR / 'Day4-TrackData-Keypoint-Pose'

if not (repo_dir / 'data.yaml').exists():
    if repo_dir.exists():
        import shutil
        shutil.rmtree(repo_dir)
    print(f'🌀 Đang git clone repository từ: {REPO_URL.strip()}...')
    res = subprocess.run(['git', 'clone', REPO_URL.strip(), str(repo_dir)], capture_output=True, text=True)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError('❌ Không thể git clone repository. Vui lòng kiểm tra lại REPO_URL (đảm bảo repo public và đúng URL).')
    print('✅ Git clone repository thành công!')

LAB_ROOT = repo_dir.resolve()
os.chdir(str(LAB_ROOT))

TRAIN_IMAGES = LAB_ROOT / 'dataset/images/train'
TRAIN_LABELS = LAB_ROOT / 'dataset/labels/train'
TEST_IMAGES = LAB_ROOT / 'dataset/images/test'
TEST_LABELS = LAB_ROOT / 'dataset/labels/test'
OUTPUTS = LAB_ROOT / 'outputs'
RUNS = OUTPUTS / 'runs'
GOLD_DIR = LAB_ROOT / 'instructor/gold'
OUTPUTS.mkdir(parents=True, exist_ok=True)
TRAIN_LABELS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LAB_ROOT / 'tools'))

# =====================================================================
# 2. Upload & Nạp file Golden Set (gold_labels.zip do Instructor cấp)
# =====================================================================
def find_gold_zip():
    for z in list(CONTENT_DIR.glob('*.zip')) + list(Path.cwd().glob('*.zip')):
        if 'gold' in z.name.lower() and zipfile.is_zipfile(z):
            return z
    return None

gold_zip = find_gold_zip()

if not (GOLD_DIR / 'labels/train').exists():
    if gold_zip:
        print(f'🏆 Đang nạp Golden Set ZIP: {gold_zip.name} -> {GOLD_DIR}')
        GOLD_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(gold_zip) as archive:
            archive.extractall(GOLD_DIR)
    elif CONTENT_DIR == Path('/content'):
        try:
            from google.colab import files
            print('\n💡 Hãy chọn upload file ZIP Golden Set (gold_labels.zip) do Instructor cấp:')
            uploaded = files.upload()
            for name, payload in uploaded.items():
                z_path = CONTENT_DIR / name
                z_path.write_bytes(payload)
                if zipfile.is_zipfile(z_path):
                    GOLD_DIR.mkdir(parents=True, exist_ok=True)
                    with zipfile.ZipFile(z_path) as archive:
                        archive.extractall(GOLD_DIR)
                    break
        except Exception as e:
            print('Colab Gold Set upload error:', e)

# Sinh data_colab.yaml trỏ path chính xác
config = yaml.safe_load((LAB_ROOT / 'data.yaml').read_text(encoding='utf-8'))
config['path'] = str(LAB_ROOT)
DATA_YAML = LAB_ROOT / 'data_colab.yaml'
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding='utf-8')

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'yolo26n-pose.pt'

print('\n--- KẾT QUẢ KHỞI TẠO MÔI TRƯỜNG ---')
print('LAB_ROOT :', LAB_ROOT)
print('device   :', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU - khuyến nghị dùng GPU T4 trên Colab')
print('kpt_shape:', config['kpt_shape'], '| flip_idx:', config['flip_idx'])
print('gold_set :', (GOLD_DIR / 'labels/train').exists())


## 1. Kiểm nhãn học viên & Đánh giá với Gold Set (`eval_vs_gold.json`)

Chạy script kiểm tra 20 file nhãn YOLO Pose (`train_01.txt` .. `train_20.txt`) và đánh giá OKS so với Gold set.


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

train_files = sorted(list(TRAIN_LABELS.glob('train_*.txt')))

# 1.1 Check định dạng nhãn train
result = subprocess.run(
    [sys.executable, str(LAB_ROOT / 'tools/check_pose_labels.py'),
     '--images', str(TRAIN_IMAGES), '--labels', str(TRAIN_LABELS)],
    capture_output=True, text=True)
print(result.stdout or result.stderr)
assert result.returncode == 0, 'Nhãn train chưa đạt định dạng hoặc chưa đủ 20 ảnh (train_01..train_20). Vui lòng kiểm tra lại repo GitHub của bạn.'

# Ghi kết quả check nhãn ra file log
(OUTPUTS / 'check_labels_output.txt').write_text(result.stdout or result.stderr, encoding='utf-8')

# 1.2 Đánh giá OKS nhãn học viên vs Gold set (nếu có gold set)
gold_labels_path = GOLD_DIR / 'labels/train'
if gold_labels_path.exists():
    print('\n🏆 Đang đánh giá nhãn học viên vs Golden Set...')
    eval_cmd = [
        sys.executable, str(LAB_ROOT / 'tools' / 'evaluate_pose_annotations.py'),
        '--student', str(TRAIN_LABELS),
        '--gold', str(gold_labels_path),
        '--out', str(OUTPUTS / 'eval_vs_gold.json')
    ]
    eval_res = subprocess.run(eval_cmd, capture_output=True, text=True)
    print(eval_res.stdout or eval_res.stderr)
    if (OUTPUTS / 'eval_vs_gold.json').exists():
        print(f'✅ Đã ghi kết quả đánh giá OKS: {OUTPUTS / "eval_vs_gold.json"}')

# 1.3 Sinh Visibility Report
vis_cmd = [
    sys.executable, str(LAB_ROOT / 'tools' / 'visibility_report.py'),
    '--labels', str(TRAIN_LABELS),
    '--out-json', str(OUTPUTS / 'visibility_report.json'),
    '--out-md', str(LAB_ROOT / 'reports' / 'visibility_report.md')
]
subprocess.run(vis_cmd, capture_output=True, text=True)
print('✅ Đã ghi visibility report: outputs/visibility_report.json & reports/visibility_report.md')

from poselib import parse_yolo_pose_file

def count(label_dir):
    files = sorted(label_dir.glob('*.txt'))
    people = [p for f in files for p in parse_yolo_pose_file(f)]
    flags = [v for p in people for _, _, v in p.keypoints]
    return len(files), len(people), flags.count(2), flags.count(1), flags.count(0)

for name, directory in (('train (nhãn học viên)', TRAIN_LABELS), ('test  (nhãn phát sẵn)', TEST_LABELS)):
    files, people, v2, v1, v0 = count(directory)
    print(f'{name}: {files} file, {people} skeleton, v=2 {v2}, v=1 {v1}, v=0 {v0}')


## 2. Model gốc, chưa học gì từ nhãn của bạn

`yolo26n-pose.pt` đã được train sẵn trên COCO - tức là nó **đã biết** bộ 17 điểm này.
Đo nó trên tập test trước, để lát nữa có mốc mà so. Đây là baseline, không phải đối thủ.


In [ ]:
baseline_model = YOLO(MODEL_NAME)
baseline_metrics = baseline_model.val(
    data=str(DATA_YAML), split='val', device=DEVICE, plots=False,
    project=str(RUNS), name='baseline', exist_ok=True)

def summarize(metrics):
    return {
        'pose_mAP50': round(float(metrics.pose.map50), 4),
        'pose_mAP50_95': round(float(metrics.pose.map), 4),
        'pose_precision': round(float(metrics.pose.mp), 4),
        'pose_recall': round(float(metrics.pose.mr), 4),
        'box_mAP50': round(float(metrics.box.map50), 4),
        'box_mAP50_95': round(float(metrics.box.map), 4),
    }

baseline = summarize(baseline_metrics)
print(json.dumps(baseline, indent=2))


## 3. Fine-tune trên nhãn của bạn

`fliplr=0.5` là augmentation lật ảnh ngang. Ultralytics đọc `flip_idx` trong `data.yaml`
để đổi tên khớp trái/phải cho ảnh lật. Đây chính là slide 15: nếu nhãn của bạn đảo trái/phải,
model được dạy cái sai đó **hai lần** - một lần ở ảnh gốc, một lần ở ảnh lật.


In [ ]:
model = YOLO(MODEL_NAME)
train_result = model.train(
    data=str(DATA_YAML),
    epochs=80,
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=2,
    project=str(RUNS),
    name='pose_finetune',
    exist_ok=True,
    seed=20260915,
    fliplr=0.5,
    plots=True,
    patience=30,
)
BEST = Path(model.trainer.best)
print('checkpoint tốt nhất:', BEST)


## 4. Đánh giá trên đúng tập test (`eval_model.json`)

`data.yaml` trỏ `val` tới 10 ảnh test đã có nhãn phát sẵn. Không đổi nó sang train.


In [ ]:
best_model = YOLO(str(BEST))
finetuned_metrics = best_model.val(
    data=str(DATA_YAML), split='val', device=DEVICE, plots=True,
    project=str(RUNS), name='pose_eval', exist_ok=True)
finetuned = summarize(finetuned_metrics)

comparison = {'baseline_' + MODEL_NAME.replace('.pt', ''): baseline, 'finetuned': finetuned,
              'delta': {k: round(finetuned[k] - baseline[k], 4) for k in baseline}}
(OUTPUTS / 'eval_model.json').write_text(json.dumps(comparison, ensure_ascii=False, indent=2))

header = f"{'chỉ số':<16}{'gốc':>10}{'sau fine-tune':>16}{'chênh':>10}"
print(header); print('-' * len(header))
for key in baseline:
    print(f'{key:<16}{baseline[key]:>10.4f}{finetuned[key]:>16.4f}{comparison["delta"][key]:>+10.4f}')
print('\n✅ Đã ghi outputs/eval_model.json')


## 5. Nhìn tận mắt - 10 ảnh test kèm pose model đoán


In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image

PRED_DIR = RUNS / 'predictions'
best_model.predict(source=str(TEST_IMAGES), conf=0.25, device=DEVICE, save=True,
                   project=str(PRED_DIR), name='test', exist_ok=True)
predicted = sorted((PRED_DIR / 'test').glob('*.jpg'))

columns = 5
rows = math.ceil(len(predicted) / columns)
figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.2 * rows))
for axis, path in zip(axes.flat, predicted):
    axis.imshow(Image.open(path)); axis.set_title(path.stem, fontsize=9)
for axis in axes.flat:
    axis.axis('off')
plt.tight_layout(); plt.show()


## 6. Nhãn của bạn vs model chấm thử, trên chính tập train


In [ ]:
from poselib import Person, greedy_match, image_size

OUT_OF_FRAME = 0
rows_out = []
for image_path in sorted(TRAIN_IMAGES.glob('*.jpg')):
    width, height = image_size(image_path)
    mine = parse_yolo_pose_file(TRAIN_LABELS / f'{image_path.stem}.txt')
    prediction = best_model.predict(source=str(image_path), conf=0.25, device=DEVICE, verbose=False)[0]
    theirs = []
    for box, keypoints in zip(prediction.boxes.xywhn.tolist(), prediction.keypoints.data.tolist()):
        points = [(x / width, y / height, 2 if score >= 0.5 else 1) if score > 0.05 else (0.0, 0.0, OUT_OF_FRAME)
                  for x, y, score in keypoints]
        theirs.append(Person(tuple(box), points))
    matched, extra, missed = greedy_match(theirs, mine, width, height)
    for _, _, score in matched:
        rows_out.append((image_path.stem, round(score, 3)))
    if extra or missed:
        rows_out.append((image_path.stem + ' (số người lệch)', f'model {len(theirs)} / bạn {len(mine)}'))

print(f"{'ảnh':<28}{'OKS model vs nhãn của bạn':>28}")
for stem, value in sorted(rows_out, key=lambda r: (isinstance(r[1], str), r[1])):
    print(f'{stem:<28}{value:>28}')
print('\nOKS thấp = model và bạn bất đồng. Xem lại ảnh đó bằng tools/visualize_pose.py.')


## 7. Kiểm tra 9 Deliverables & Đóng gói `outputs.zip` để tải về Colab

Cell này kiểm tra toàn bộ yêu cầu bài nộp và tự động đóng gói các file kết quả (`eval_vs_gold.json`, `eval_model.json`, `visibility_report.json`, `visibility_report.md`,...) thành file `outputs.zip` để học viên tải về máy local phục vụ viết `reports/REPORT.md`.


In [ ]:
import shutil
import zipfile

# 7.1 Kiểm tra 9 Deliverables của bộ nộp Lane S
checklist = [
    ("1. Nhãn YOLO Pose (20 txt)", len(list(TRAIN_LABELS.glob("train_*.txt"))) == 20),
    ("2. COCO Export JSON", (LAB_ROOT / "annotations/coco_keypoints/person_keypoints_default.json").exists() or (LAB_ROOT / "person_keypoints_default.json").exists()),
    ("3. Face & Hand annotations", (LAB_ROOT / "annotations/face_hand").exists()),
    ("4. Reports visibility_report.md", (LAB_ROOT / "reports/visibility_report.md").exists()),
    ("5. Outputs visibility_report.json", (OUTPUTS / "visibility_report.json").exists()),
    ("6. GUIDELINE_MINI.md", (LAB_ROOT / "GUIDELINE_MINI.md").exists()),
    ("7. Outputs eval_vs_gold.json", (OUTPUTS / "eval_vs_gold.json").exists()),
    ("8. Outputs eval_model.json", (OUTPUTS / "eval_model.json").exists()),
    ("9. Reports REPORT.md", (LAB_ROOT / "reports/REPORT.md").exists()),
]

print("=== CHECKLIST YÊU CẦU BÀI NỘP (9 DELIVERABLES) ===")
for name, status in checklist:
    icon = "✅ PASS" if status else "⚠️ PENDING"
    print(f"{name:<45} {icon}")

# 7.2 Đóng gói các file outputs cần thiết vào outputs.zip
zip_outputs_path = CONTENT_DIR / "outputs.zip"
with zipfile.ZipFile(zip_outputs_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for f in OUTPUTS.rglob("*"):
        if f.is_file() and not 'runs' in f.parts:
            arcname = Path("outputs") / f.relative_to(OUTPUTS)
            zipf.write(f, arcname)
    if (LAB_ROOT / "reports/visibility_report.md").exists():
        zipf.write(LAB_ROOT / "reports/visibility_report.md", Path("reports/visibility_report.md"))

print(f"\n📦 Đã đóng gói thành công: {zip_outputs_path} ({zip_outputs_path.stat().st_size} bytes)")

# 7.3 Tự động kích hoạt tải về trên Colab
try:
    from google.colab import files
    print("⬇️ Đang kích hoạt tải về file outputs.zip...")
    files.download(str(zip_outputs_path))
except ImportError:
    print(f"Local environment: File đã tạo tại {zip_outputs_path.resolve()}")


## Câu hỏi phân tích để nộp

Trả lời trong `reports/REPORT.md`:

1. `pose_mAP50-95` thay đổi bao nhiêu sau fine-tune? Nếu nó **giảm**, hãy giải thích:
   20 ảnh của bạn dạy được model điều gì mà COCO chưa dạy, và nó làm hỏng điều gì?
2. `box_mAP` và `pose_mAP` chênh nhau bao nhiêu? Model tìm *người* dễ hơn hay tìm *khớp* dễ hơn?
3. Ở mục 5, tìm một ảnh model đoán sai. Gọi tên lỗi theo bốn loại của slide 43.
4. Ở mục 6, ảnh nào có OKS thấp nhất giữa bạn và model? Ai đúng - và bạn dựa vào đâu để nói vậy?
5. Trong `tools/evaluate_pose_annotations.py` bạn đã có OKS nhãn-của-bạn vs gold.
   Ảnh nào bạn gán tệ nhất *cũng* là ảnh model đoán tệ nhất? Nếu có, điều đó nói gì về ảnh đó?
